In [21]:
# News EDA
# load and setup path and required libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve

# load the dataset
path = "..\\data\\main_news.parquet"
df = pd.read_parquet(path)
# check the structure of the datasetdisplay(df.head())

# changing the data into dataframe

def create_dataframe(data):
    df = pd.DataFrame(data)
    return df

# create a dataframe from the loaded data
df = create_dataframe(df)
# check the structure of the dataframe
display(df.head())

# check for more than 1 news of a particular ticker on a particular date
df["time_published"] = pd.to_datetime(df["time_published"], errors="coerce")
df["ticker_date"] = df["symbol"].astype(str) + "_" + df["time_published"].dt.strftime("%Y-%m-%d")
duplicate_rows = df[df.duplicated(subset=["ticker_date"], keep=False)]
display(duplicate_rows[["symbol", "time_published", "title","overall_sentiment_score"]])

,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,symbol
0,Blood-based gene expression signatures of medication-free outpatients with major depressive disorder: integrative genome-wide and candidate gene analyses,https://www.nature.com/articles/srep18776,2016-01-05 05:51:32,"[Hiroaki Hori, Daimei Sasayama, Toshiya Teraishi, Noriko Yamamoto, Seiji Nakamura, Miho Ota, Kotaro Hattori, Yoshiharu Kim, Teruhiko Higuchi, Hiroshi Kunugi]","This study investigates blood-based gene expression signatures in medication-free outpatients with major depressive disorder (MDD) compared to healthy controls. It identifies differentially expressed genes (DEGs) that map to significantly over-represented pathways like ""synaptic transmission"" and reveals a disrupted co-expression network of candidate genes in MDD patients, highlighting key molecules and networks involved in depression. The research suggests that candidate genes, often overlooked by stringent statistical thresholds, are valuable targets for depression research.",NaN,Nature,General,Nature,"[{'topic': 'life_sciences', 'relevance_score': '0.918616'}]",0.122718,Neutral,"[{'ticker': 'A', 'relevance_score': '0.900629', 'ticker_sentiment_score': '0.101134', 'ticker_sentiment_label': 'Neutral'}]",A
1,Agilent accuses Twist Bioscience CEO of stealing trade secrets,https://www.biopharmadive.com/news/agilent-accuses-twist-bioscience-ceo-of-stealing-trade-secrets/413405/,2016-02-05 05:51:32,[Nicole Gray],"Agilent Technologies has filed a lawsuit accusing Twist Bioscience CEO, Emily Leproust, of stealing trade secrets related to synthetic gene manufacturing when she left Agilent to co-found Twist. Agilent alleges that Twist's financial success is due to this stolen proprietary gene technology. Twist Bioscience, which has raised over $130 million in venture funding, calls these claims ""baseless and without merit"" and plans to vigorously defend itself.",NaN,BioPharma Dive,General,BioPharma Dive,"[{'topic': 'life_sciences', 'relevance_score': '0.905912'}, {'topic': 'technology', 'relevance_score': '0.833628'}]",-0.638520,Bearish,"[{'ticker': 'A', 'relevance_score': '1.000000', 'ticker_sentiment_score': '-0.639674', 'ticker_sentiment_label': 'Bearish'}]",A
2,Avago adopts new name,https://www.coloradoan.com/story/money/2016/02/16/avago-adopts-new-name/80450916/,2016-02-16 05:51:32,[Adrian D. Garcia],"Avago Technologies, a major employer in Fort Collins, has rebranded to Broadcom Ltd. following a $37 billion acquisition. Avago acquired Broadcom and adopted its name, with Avago CEO Hock Tan remaining in his leadership role. The company plans to release its first-quarter financial report on March 3.",NaN,The Coloradoan,General,The Coloradoan,"[{'topic': 'mergers_and_acquisitions', 'relevance_score': '1.000000'}, {'topic': 'technology', 'relevance_score': '0.813530'}, {'topic': 'earnings', 'relevance_score': '0.722721'}, {'topic': 'manufacturing', 'relevance_score': '0.633771'}]",0.176532,Somewhat-Bullish,"[{'ticker': 'A', 'relevance_score': '0.603131', 'ticker_sentiment_score': '-0.119860', 'ticker_sentiment_label': 'Neutral'}, {'ticker': 'AVGO', 'relevance_score': '1.000000', 'ticker_sentiment_score': '0.440914', 'ticker_sentiment_label': 'Bullish'}]",A
3,"Nanotechnology Company Exicure Raises Over $42 Million from Bill Gates, AbbVie, Agilent and Others",https://www.biospace.com/nanotechnology-company-exicure-raises-over-42-million-from-b-bill-gates-b-abbvie-agilent-and-others,2016-02-26 05:51:32,[Mark Terry],"Exicure, a nanotechnology company formerly known as AuraSense Therapeutics, has raised over $40.75 million in equity funding, including investments from prominent figures like Bill Gates and companies like AbbVie and Agilent. The company specializes in developing three-dimensional, spherical nucleic acid (SNA) architecture for immunomodulatory and gene silencing drugs, with a

,symbol,time_published,title,overall_sentiment_score
16,A,2017-07-26 09:44:00,"Agilent Tech to invest S$85m over 5 years in manufacturing, research in Singapore",0.536373
17,A,2017-07-26 21:50:00,Agilent to invest S$85m over 5 years in Singapore,0.403981
23,A,2017-10-31 03:46:22,Hewlett-Packard Archive Destroyed,0.113391
24,A,2017-10-31 09:00:00,"HP founder's archive"" which is part of the history of Silicon Valley burned down in a large fire",-0.139867
45,A,2019-01-01 00:00:00,"Stocks to Watch: Macy’s, McDonald’s, Agilent, CVS, GoPro, Tilray, Stanley Black & Decker, Facebook, Children’s Place",-0.064228
...,...,...,...,...
770613,ZTS,2025-12-24 03:09:03,The Technical Signals Behind (ZTS) That Institutions Follow,0.062804
770614,ZTS,2025-12-24 11:01:04,Albert White Bought 4.4% More Shares In Cooper Companies,0.237106
770615,ZTS,2025-12-24 16:30:00,Zoetis Inc. stock outperforms competitors on strong trading day,0.390869
770616,ZTS,2025-12-24 17:30:00,Prescription for Growth: Why Zoetis and Viatris are Defining Healthcare’s New Winners in 2025,0.276550


In [9]:
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

display(df.head())

,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment,symbol
0,Blood-based gene expression signatures of medication-free outpatients with major depressive disorder: integrative genome-wide and candidate gene analyses,https://www.nature.com/articles/srep18776,2016-01-05 05:51:32,"[Hiroaki Hori, Daimei Sasayama, Toshiya Teraishi, Noriko Yamamoto, Seiji Nakamura, Miho Ota, Kotaro Hattori, Yoshiharu Kim, Teruhiko Higuchi, Hiroshi Kunugi]","This study investigates blood-based gene expression signatures in medication-free outpatients with major depressive disorder (MDD) compared to healthy controls. It identifies differentially expressed genes (DEGs) that map to significantly over-represented pathways like ""synaptic transmission"" and reveals a disrupted co-expression network of candidate genes in MDD patients, highlighting key molecules and networks involved in depression. The research suggests that candidate genes, often overlooked by stringent statistical thresholds, are valuable targets for depression research.",NaN,Nature,General,Nature,"[{'topic': 'life_sciences', 'relevance_score': '0.918616'}]",0.122718,Neutral,"[{'ticker': 'A', 'relevance_score': '0.900629', 'ticker_sentiment_score': '0.101134', 'ticker_sentiment_label': 'Neutral'}]",A
1,Agilent accuses Twist Bioscience CEO of stealing trade secrets,https://www.biopharmadive.com/news/agilent-accuses-twist-bioscience-ceo-of-stealing-trade-secrets/413405/,2016-02-05 05:51:32,[Nicole Gray],"Agilent Technologies has filed a lawsuit accusing Twist Bioscience CEO, Emily Leproust, of stealing trade secrets related to synthetic gene manufacturing when she left Agilent to co-found Twist. Agilent alleges that Twist's financial success is due to this stolen proprietary gene technology. Twist Bioscience, which has raised over $130 million in venture funding, calls these claims ""baseless and without merit"" and plans to vigorously defend itself.",NaN,BioPharma Dive,General,BioPharma Dive,"[{'topic': 'life_sciences', 'relevance_score': '0.905912'}, {'topic': 'technology', 'relevance_score': '0.833628'}]",-0.638520,Bearish,"[{'ticker': 'A', 'relevance_score': '1.000000', 'ticker_sentiment_score': '-0.639674', 'ticker_sentiment_label': 'Bearish'}]",A
2,Avago adopts new name,https://www.coloradoan.com/story/money/2016/02/16/avago-adopts-new-name/80450916/,2016-02-16 05:51:32,[Adrian D. Garcia],"Avago Technologies, a major employer in Fort Collins, has rebranded to Broadcom Ltd. following a $37 billion acquisition. Avago acquired Broadcom and adopted its name, with Avago CEO Hock Tan remaining in his leadership role. The company plans to release its first-quarter financial report on March 3.",NaN,The Coloradoan,General,The Coloradoan,"[{'topic': 'mergers_and_acquisitions', 'relevance_score': '1.000000'}, {'topic': 'technology', 'relevance_score': '0.813530'}, {'topic': 'earnings', 'relevance_score': '0.722721'}, {'topic': 'manufacturing', 'relevance_score': '0.633771'}]",0.176532,Somewhat-Bullish,"[{'ticker': 'A', 'relevance_score': '0.603131', 'ticker_sentiment_score': '-0.119860', 'ticker_sentiment_label': 'Neutral'}, {'ticker': 'AVGO', 'relevance_score': '1.000000', 'ticker_sentiment_score': '0.440914', 'ticker_sentiment_label': 'Bullish'}]",A
3,"Nanotechnology Company Exicure Raises Over $42 Million from Bill Gates, AbbVie, Agilent and Others",https://www.biospace.com/nanotechnology-company-exicure-raises-over-42-million-from-b-bill-gates-b-abbvie-agilent-and-others,2016-02-26 05:51:32,[Mark Terry],"Exicure, a nanotechnology company formerly known as AuraSense Therapeutics, has raised over $40.75 million in equity funding, including investments from prominent figures like Bill Gates and companies like AbbVie and Agilent. The company specializes in developing three-dimensional, spherical nucleic acid (SNA) architecture for immunomodulatory and gene silencing drugs, with a

In [12]:
df.shape

(770623, 14)